# 🐼 Séance 4 — Pandas : regrouper, croiser, combiner, dater · Partie 1 : pratiquer
_Master IA‑GI — Python pour les sciences de données_

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yelallioui/python-for-ds-s04/blob/main/tp-seance-04.ipynb)

**Comment ouvrir ce notebook :** sur votre ordinateur, ouvrez-le **depuis son dossier** `seance-04/`, sans le déplacer : il lit le Titanic dans `mini-projet-04/data/titanic.csv`, à côté de lui. Pandas doit être installé (`pip install pandas` ; déjà présent avec Anaconda).

**Dans Colab**, le notebook arrive **seul** : exécutez la cellule **Étape 0** (plus bas) pour récupérer le dossier du TP, avec `titanic.csv`. Pour **garder votre travail**, faites **Fichier ▸ Enregistrer une copie dans Drive** (ou dans GitHub) : Colab ne sauvegarde pas un notebook ouvert depuis GitHub.

**Objectifs** — à la fin de ce notebook, vous saurez :
- **regrouper** avec `groupby` (une ou deux clés, `agg`, `size` et `count`, `pd.cut`) et **lire l'effectif** à côté du taux
- **croiser** deux variables avec `pivot_table` et `crosstab`, et passer du format large au format long (`melt`, `pivot`)
- **combiner** des tableaux : `concat` pour empiler ou aligner, `merge` pour joindre, en **vérifiant les clés**
- **dater** : convertir du texte en dates, découper par période, `resample`, `shift`, `rolling`
- **appliquer** une fonction à bon escient (`map`, `apply`, `transform`)

**Durée estimée** : 2 heures environ · **Parties** :
- **A** · Regrouper
- **B** · Croiser
- **C** · Combiner
- **D** · Dater
- **E** · Appliquer

**Comment travailler seul**
1. Exécutez les cellules **dans l'ordre** (`Maj + Entrée`).
2. Lisez le *cours en bref*, puis **exécutez le code complet** : c'est lui qui contient les détails des diapos.
3. Faites chaque exercice en remplaçant les `...` par votre code, puis exécutez la cellule : un ✅ ou un ❌ s'affiche.
4. Un ❌ ? Ouvrez l'**indice 1**, puis 2, puis 3.

### 🧰 Pour commencer : importer, puis se contrôler
Exécutez la cellule ci-dessous. Elle importe ce qu'il faut et définit **`verifier`**, la fonction qui affiche ✅ ou ❌ sous chaque exercice. Rien de nouveau dans son écriture : une fonction (`def`), un `try` / `except` et un `lambda`, vus en séance 1.

In [ ]:
import numpy as np
import pandas as pd

print("Pandas", pd.__version__, "· NumPy", np.__version__)

def verifier(ex, description, test):
    """Affiche ✅ si test() est vrai, ❌ sinon (même si le test provoque une erreur)."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + ex + " · " + description)

### 🪪 Qui êtes-vous ?
Écrivez votre **nom** et votre **prénom** **entre guillemets**, en remplaçant les `...` (par exemple `nom = "Benali"`). Ces deux lignes me permettent de reconnaître votre travail : le contrôle reste ❌ tant qu'elles ne sont pas remplies. **Votre dépôt est public : n'y écrivez jamais de donnée personnelle (CNE, mot de passe, téléphone).**

In [ ]:
nom = ...
prenom = ...

print(nom, prenom)
verifier("identité", "nom et prénom renseignés", lambda: type(nom) is str and type(prenom) is str and len(nom.strip()) >= 2 and len(prenom.strip()) >= 2)

### 📁 Étape 0 · Récupérer le dossier du TP (dans Colab)
Dans **Colab**, le notebook arrive **seul** : le fichier `titanic.csv` **reste sur GitHub**. Exécutez la cellule ci-dessous : elle **télécharge le dossier du TP** dans la session Colab, puis s'y place. Le fichier se trouve alors dans `python-for-ds-s04/mini-projet-04/data/titanic.csv` (panneau **📁** à gauche, bouton **↻** pour actualiser).

**Sur votre ordinateur**, cette cellule ne fait rien : ouvrez simplement le notebook depuis son dossier.

Les fichiers d'une session Colab sont **temporaires** : ils disparaissent à la fermeture. Pour garder un fichier, par exemple `mini-projet-04/result/s4_synthese.csv` : clic droit dessus ▸ **Télécharger**.

In [ ]:
import os
import subprocess

try:
    import google.colab                 # ce module n'existe que dans Colab
    dans_colab = True
except ImportError:
    dans_colab = False

if dans_colab and not os.path.exists("mini-projet-04/data/titanic.csv"):
    subprocess.run(["git", "clone", "-q", "https://github.com/yelallioui/python-for-ds-s04"])
    os.chdir("python-for-ds-s04/")

print("dossier de travail :", os.getcwd())
print("mini-projet-04/data/titanic.csv trouvé :", os.path.exists("mini-projet-04/data/titanic.csv"))

---
# Partie 1 — Pratiquer : regrouper, croiser, combiner, dater
_Environ 2 heures · cinq parties (A à E) · 24 exercices directs._

Chaque exercice pose une **question directe** : vous écrivez le code, vous **affichez le résultat avec `print`**, et un ✅ ou un ❌ vous dit si c'est juste. Gardez le réflexe : **en cas de doute, `print` !**

Cette partie travaille sur le **Titanic brut** (avec ses trous : c'est voulu) et sur deux petits jeux fabriqués. Le **mini-projet** reprendra les mêmes gestes sur **votre tableau propre** de la séance 3.

### Les données de la séance
- **Titanic** (891 passagers) : le fil rouge, lu dans `mini-projet-04/data/titanic.csv`. **Ne le nettoyez pas ici** : les trous (`NaN`) font partie des exercices.
- **Deux tables** `pas` (passagers) et `bil` (billets), obtenues **en coupant le Titanic** : la table des billets est **incomplète à dessein** (19 passagers n'ont pas de billet).
- **Des ventes quotidiennes** de 90 jours, **fabriquées par une formule** : tout le monde obtient les mêmes chiffres.

In [ ]:
df = pd.read_csv("mini-projet-04/data/titanic.csv")

# deux tables obtenues en coupant le fichier ; la table des billets perd un passager sur quarante-cinq
pas = df[["PassengerId", "Name", "Sex", "Age", "Pclass", "Survived"]]
bil = df[["PassengerId", "Ticket", "Fare", "Cabin", "Embarked"]]
bil = bil[bil["PassengerId"] % 45 != 0]
print(df.shape, pas.shape, bil.shape)

---
## Partie A — Regrouper
_Diapos 8 à 15._

**Cours en bref**
- `groupby` suit **trois temps** : **séparer** les lignes en groupes selon une clé, **appliquer** une statistique à chaque groupe, **combiner** les résultats dans une Series (ou un DataFrame) dont l'**index est la clé**.
- Lire une écriture : `df.groupby("Pclass")["Survived"].mean()` = *la clé*, *la colonne*, *la statistique*. `Survived` vaut 1 ou 0 : sa moyenne est une **proportion**.
- `agg` calcule **plusieurs statistiques** : `agg(age_moyen=("Age", "mean"), tarif_median=("Fare", "median"))`.
- **Deux clés** : `groupby(["Pclass", "Embarked"]).size()` donne un index à **deux niveaux** ; `unstack()` le met en tableau.
- `size()` compte les **lignes** du groupe, `count()` les **valeurs présentes** d'une colonne : les `NaN` font la différence.
- `pd.cut(df["Age"], bornes, labels=noms)` range une valeur numérique en **tranches** ; avec une colonne de catégories, on écrit `groupby(..., observed=True)`.
- **Lisez toujours l'effectif** à côté du taux : `agg(["mean", "size"])`.

In [ ]:
print(df.groupby("Pclass")["Survived"].mean().round(3))                 # la clé, la colonne, la statistique
print(df.groupby("Pclass").agg(age_moyen=("Age", "mean"), tarif_median=("Fare", "median")).round(2))
print(df.groupby(["Pclass", "Embarked"]).size().unstack())              # deux clés, puis un tableau

In [ ]:
print(df.groupby("Pclass").size().tolist(), df.groupby("Pclass")["Age"].count().tolist())   # lignes, puis âges présents

bornes = [0, 12, 18, 60, 80]
noms = ["Enfant", "Adolescent", "Adulte", "Senior"]
tranche = pd.cut(df["Age"], bornes, labels=noms)                        # une tranche par passager (NaN si l'âge manque)
print(df.groupby(tranche, observed=True)["Survived"].agg(["mean", "size"]).round(3))

**✏️ Exercice A1 — La survie par classe**

`survie_classe` : le **taux de survie de chaque classe** (une Series indexée par `Pclass`), avec `groupby`.

<details>
<summary>💡 Indice 1 · la piste</summary>

Trois pièces : la clé, la colonne, la statistique.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

La clé est `Pclass`, la colonne `Survived`, la statistique `mean()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`survie_classe = df.groupby("Pclass")["Survived"].mean()` : 0,630 ; 0,473 ; 0,242.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
survie_classe = ...

verifier("A1", "une Series de trois classes", lambda: isinstance(survie_classe, pd.Series) and list(survie_classe.index) == [1, 2, 3])
verifier("A1", "63 %, 47 % et 24 %", lambda: np.allclose(survie_classe.to_numpy(), [0.630, 0.473, 0.242], atol=0.001))

**✏️ Exercice A2 — Plusieurs statistiques**

`stats` : un DataFrame, indexé par `Pclass`, avec deux colonnes : `age_moyen` (l'âge **moyen**) et `tarif_median` (le tarif **médian**), calculées avec `agg` et des **paires nommées**.

<details>
<summary>💡 Indice 1 · la piste</summary>

`agg(nom=(colonne, fonction), ...)`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df.groupby("Pclass").agg(age_moyen=("Age", "mean"), tarif_median=("Fare", "median"))`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

La première classe : 38,23 ans en moyenne, 60,29 de tarif médian.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
stats = ...

verifier("A2", "deux colonnes nommées, trois classes", lambda: isinstance(stats, pd.DataFrame) and list(stats.columns) == ["age_moyen", "tarif_median"] and stats.shape == (3, 2))
verifier("A2", "âge moyen de la 1re classe : 38,23", lambda: abs(stats.loc[1, "age_moyen"] - 38.23) < 0.01)
verifier("A2", "tarif médian de la 3e classe : 8,05", lambda: abs(stats.loc[3, "tarif_median"] - 8.05) < 0.01)

**✏️ Exercice A3 — Deux clés, puis un tableau**

`effectifs` : le **tableau** du nombre de passagers, avec les **classes en lignes** et les **ports en colonnes** (`groupby` sur deux clés, `size`, puis `unstack`).

<details>
<summary>💡 Indice 1 · la piste</summary>

Une liste de deux clés dans `groupby` ; `size()` compte les lignes ; `unstack()` met le deuxième niveau en colonnes.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df.groupby(["Pclass", "Embarked"]).size().unstack()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

La case la plus pleine : 353 passagers de 3e classe embarqués à `S`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
effectifs = ...

verifier("A3", "un tableau de 3 lignes et 3 colonnes", lambda: isinstance(effectifs, pd.DataFrame) and effectifs.shape == (3, 3))
verifier("A3", "353 passagers de 3e classe au port S", lambda: effectifs.loc[3, "S"] == 353)
verifier("A3", "889 passagers ont un port (2 manquent)", lambda: effectifs.to_numpy().sum() == 889)

**✏️ Exercice A4 — size contre count**

1. `n_lignes` : le nombre de **lignes** de chaque classe
2. `n_ages` : le nombre d'**âges présents** de chaque classe
3. `n_sans_age` : la différence (les âges manquants de chaque classe)

<details>
<summary>💡 Indice 1 · la piste</summary>

`size()` compte les lignes ; `count()` sur une colonne compte les valeurs présentes.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df.groupby("Pclass").size()` ; `df.groupby("Pclass")["Age"].count()` ; puis la soustraction.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

136 âges manquent en 3e classe, 177 en tout.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
n_lignes = ...
n_ages = ...
n_sans_age = ...

verifier("A4", "216, 184 et 491 lignes", lambda: list(n_lignes) == [216, 184, 491])
verifier("A4", "186, 173 et 355 âges présents", lambda: list(n_ages) == [186, 173, 355])
verifier("A4", "136 âges manquants en 3e classe, 177 en tout", lambda: n_sans_age[3] == 136 and n_sans_age.sum() == 177)

**✏️ Exercice A5 — Des tranches d'âge**

Avec `bornes` et `noms` (cellule de cours) :
1. `tranche` : l'âge rangé en tranches avec `pd.cut`
2. `survie_tranche` : le **taux de survie de chaque tranche** (`groupby` sur la colonne `tranche`, avec `observed=True`)

<details>
<summary>💡 Indice 1 · la piste</summary>

`pd.cut(valeurs, bornes, labels=noms)` ; on peut grouper avec une Series à la place d'un nom de colonne.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`tranche = pd.cut(df["Age"], bornes, labels=noms)` ; `survie_tranche = df.groupby(tranche, observed=True)["Survived"].mean()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Les enfants : 58 % ; les seniors : 23 %.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
tranche = ...
survie_tranche = ...

verifier("A5", "quatre tranches", lambda: isinstance(survie_tranche, pd.Series) and list(survie_tranche.index) == noms)
verifier("A5", "58 % de survie chez les enfants", lambda: abs(survie_tranche["Enfant"] - 0.580) < 0.001)
verifier("A5", "23 % de survie chez les seniors", lambda: abs(survie_tranche["Senior"] - 0.227) < 0.001)

**✏️ Exercice A6 — À vous : prix par port, et effectifs**

1. `fare_port` : le **prix moyen du billet** selon le port d'embarquement
2. `resume_tranches` : pour chaque tranche d'âge, la **moyenne** de `Survived` **et l'effectif** (`agg(["mean", "size"])`)
3. `n_seniors` : le **nombre de seniors** (lu dans `resume_tranches`)

<details>
<summary>💡 Indice 1 · la piste</summary>

Le premier : la clé `Embarked`, la colonne `Fare`, la moyenne. Le deuxième : `agg` avec une liste de deux fonctions.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df.groupby("Embarked")["Fare"].mean()` ; `df.groupby(tranche, observed=True)["Survived"].agg(["mean", "size"])` ; `resume_tranches.loc["Senior", "size"]`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Cherbourg : 59,95 ; Queenstown : 13,28 ; Southampton : 27,08. Seniors : 22 passagers.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
fare_port = ...
resume_tranches = ...
n_seniors = ...

verifier("A6", "tarif moyen par port : C 59,95 · Q 13,28 · S 27,08", lambda: isinstance(fare_port, pd.Series) and np.allclose(fare_port.to_numpy(), [59.95, 13.28, 27.08], atol=0.01))
verifier("A6", "un résumé de quatre lignes avec mean et size", lambda: isinstance(resume_tranches, pd.DataFrame) and list(resume_tranches.columns) == ["mean", "size"] and len(resume_tranches) == 4)
verifier("A6", "22 seniors", lambda: n_seniors == 22)

---
## Partie B — Croiser
_Diapos 16 à 20._

**Cours en bref**
- `pd.pivot_table(df, values=..., index=..., columns=..., aggfunc=...)` croise **deux variables** : l'index en **lignes**, `columns` en **colonnes**, la statistique de `values` dans les **cellules**. C'est le *tableau croisé dynamique* d'Excel.
- `aggfunc="sum"` additionne ; `margins=True` ajoute la ligne et la colonne **All** (les totaux).
- `pd.crosstab(a, b)` **compte** les passagers par couple de valeurs ; `normalize="index"` ramène chaque ligne à **100 %**.
- `pd.melt(df, id_vars=..., value_vars=[...])` passe du format **large** (une colonne par mesure) au format **long** (une ligne par mesure) ; `pivot` fait le chemin inverse et **refuse** deux valeurs pour la même case.

In [ ]:
print(pd.pivot_table(df, values="Survived", index="Pclass", columns="Sex", aggfunc="mean").round(3))
print(pd.crosstab(df["Pclass"], df["Sex"]))

**✏️ Exercice B1 — Un tableau croisé**

1. `pt` : le **taux de survie** par classe (lignes) et par sexe (colonnes), avec `pivot_table`
2. `femme3_devant_homme1` : `True` si le taux d'une **femme de 3e classe** dépasse celui d'un **homme de 1re classe**

<details>
<summary>💡 Indice 1 · la piste</summary>

Quatre ingrédients : `values`, `index`, `columns`, `aggfunc`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`pt = pd.pivot_table(df, values="Survived", index="Pclass", columns="Sex", aggfunc="mean")` ; puis comparer `pt.loc[3, "female"]` et `pt.loc[1, "male"]`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

0,500 contre 0,369 : la réponse est `True`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
pt = ...
femme3_devant_homme1 = ...

verifier("B1", "un tableau de 3 lignes et 2 colonnes (female, male)", lambda: isinstance(pt, pd.DataFrame) and pt.shape == (3, 2) and list(pt.columns) == ["female", "male"])
verifier("B1", "50 % pour une femme de 3e classe, 37 % pour un homme de 1re", lambda: abs(pt.loc[3, "female"] - 0.5) < 0.001 and abs(pt.loc[1, "male"] - 0.369) < 0.001)
verifier("B1", "la femme de 3e classe devance l'homme de 1re", lambda: femme3_devant_homme1 is True or femme3_devant_homme1 is np.True_)

**✏️ Exercice B2 — Les totaux**

`totaux` : le tableau croisé du **nombre de survivants** (somme de `Survived`) par classe et par sexe, **avec les totaux** (`margins=True`).

<details>
<summary>💡 Indice 1 · la piste</summary>

Même tableau, avec `aggfunc="sum"` et `margins=True`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`pd.pivot_table(df, values="Survived", index="Pclass", columns="Sex", aggfunc="sum", margins=True)`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Total : 342 survivants, dont 233 femmes et 109 hommes.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
totaux = ...

verifier("B2", "4 lignes et 3 colonnes (avec All)", lambda: isinstance(totaux, pd.DataFrame) and totaux.shape == (4, 3))
verifier("B2", "342 survivants en tout", lambda: totaux.loc["All", "All"] == 342)
verifier("B2", "91 femmes survivantes en 1re classe", lambda: totaux.loc[1, "female"] == 91)

**✏️ Exercice B3 — Compter avec crosstab**

1. `ct` : le **nombre de passagers** par classe (lignes) et par sexe (colonnes), avec `crosstab`
2. `part_survie` : la **proportion de chaque ligne** pour `Survived` (0 ou 1), par classe (`normalize="index"`)

<details>
<summary>💡 Indice 1 · la piste</summary>

`pd.crosstab(a, b)` prend deux colonnes ; `normalize="index"` ramène chaque ligne à 1.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`pd.crosstab(df["Pclass"], df["Sex"])` ; `pd.crosstab(df["Pclass"], df["Survived"], normalize="index")`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

347 hommes en 3e classe ; 24 % de survivants en 3e classe.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
ct = ...
part_survie = ...

verifier("B3", "347 hommes de 3e classe", lambda: isinstance(ct, pd.DataFrame) and ct.loc[3, "male"] == 347)
verifier("B3", "chaque ligne de part_survie vaut 1", lambda: isinstance(part_survie, pd.DataFrame) and np.allclose(part_survie.sum(axis=1), 1))
verifier("B3", "24 % de survivants en 3e classe", lambda: abs(part_survie.loc[3, 1] - 0.242) < 0.001)

**✏️ Exercice B4 — Du large au long, et retour**

Avec `petit` (les trois premiers passagers) :
1. `long` : le tableau **long**, avec `melt` (colonne qui reste : `PassengerId` ; colonnes à empiler : `Age` et `Fare`)
2. `large` : le tableau **large** retrouvé avec `pivot` (index `PassengerId`, colonnes `variable`, valeurs `value`)

<details>
<summary>💡 Indice 1 · la piste</summary>

`melt` : `id_vars` reste, `value_vars` s'empilent. `pivot` : `index`, `columns`, `values`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`long = pd.melt(petit, id_vars="PassengerId", value_vars=["Age", "Fare"])` ; `large = long.pivot(index="PassengerId", columns="variable", values="value")`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`long` a 6 lignes et 3 colonnes ; `large` a 3 lignes et 2 colonnes.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
petit = df.head(3)[["PassengerId", "Age", "Fare"]]
long = ...
large = ...

verifier("B4", "6 lignes et 3 colonnes en format long", lambda: isinstance(long, pd.DataFrame) and long.shape == (6, 3) and set(long["variable"]) == {"Age", "Fare"})
verifier("B4", "3 lignes et 2 colonnes en format large", lambda: isinstance(large, pd.DataFrame) and large.shape == (3, 2))
verifier("B4", "les âges retrouvés : 22, 38, 26", lambda: np.allclose(large["Age"].to_numpy(), [22, 38, 26]))

---
## Partie C — Combiner
_Diapos 21 à 27._

**Cours en bref**
- `pd.concat([a, b], ignore_index=True)` **empile** deux tableaux de mêmes colonnes ; `ignore_index=True` renumérote de 0 à n−1.
- `pd.concat([a, b], axis=1)` place les tableaux **côte à côte** en **alignant l'index** : `join="outer"` (défaut) garde toutes les étiquettes (cases `NaN`), `join="inner"` seulement les communes.
- `pd.merge(g, d, left_on=..., right_on=..., how=...)` **joint** deux tables sur une **clé** : `inner` (correspondances), `left`, `right`, `outer`. `indicator=True` ajoute une colonne d'audit `_merge` (`both`, `left_only`, `right_only`).
- **Deux pièges** : une **clé en double** fait grossir le résultat sans erreur (`validate="one_to_one"` la détecte) ; des **clés de types différents** (entier contre texte) provoquent une erreur.
- `a.join(b, on="clé")` fusionne sur l'**index** de `b`. **Avant tout merge** : doublons de clé ? types des clés ? quel `how` ? auditer.

In [ ]:
clients = pd.DataFrame({"id": [1, 2, 3], "nom": ["Ali", "Sara", "Driss"], "ville": ["Rabat", "Casa", "Fès"]})
achats = pd.DataFrame({"client_id": [1, 1, 3, 4], "produit": ["Livre", "Stylo", "PC", "Souris"], "montant": [120, 10, 7000, 100]})
print(pd.merge(clients, achats, left_on="id", right_on="client_id", how="inner"))

full = pd.merge(pas, bil, on="PassengerId", how="left", indicator=True)      # tous les passagers, billets quand il y en a
print(full.shape, full["_merge"].value_counts().to_dict())

**✏️ Exercice C1 — Empiler**

Avec `p1` (400 passagers tirés au hasard, graine fixée) :
1. `p2` : les **autres** passagers (`df.drop(p1.index)`)
2. `tout` : `p1` et `p2` **empilés**, avec un index **renuméroté**

<details>
<summary>💡 Indice 1 · la piste</summary>

`concat` prend une liste de tableaux ; `ignore_index=True` renumérote.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`p2 = df.drop(p1.index)` ; `tout = pd.concat([p1, p2], ignore_index=True)`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`p2` a 491 lignes ; `tout` a 891 lignes et 12 colonnes.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
p1 = df.sample(400, random_state=0)
p2 = ...
tout = ...

verifier("C1", "p2 : 491 passagers", lambda: isinstance(p2, pd.DataFrame) and len(p2) == 491)
verifier("C1", "tout : 891 lignes, 12 colonnes", lambda: isinstance(tout, pd.DataFrame) and tout.shape == (891, 12))
verifier("C1", "l'index va de 0 à 890", lambda: list(tout.index[:3]) == [0, 1, 2] and tout.index[-1] == 890)

**✏️ Exercice C2 — Côte à côte**

Avec `a` et `b` (index différents) :
1. `cote_a_cote` : `a` et `b` placés **côte à côte** (toutes les étiquettes)
2. `communs` : seulement les **étiquettes communes** aux deux

<details>
<summary>💡 Indice 1 · la piste</summary>

`axis=1` pour les colonnes ; `join="inner"` pour l'intersection.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`pd.concat([a, b], axis=1)` ; `pd.concat([a, b], axis=1, join="inner")`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

4 lignes et 2 `NaN` pour le premier ; 2 lignes pour le second.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
a = pd.DataFrame({"x": [1, 2, 3]}, index=[0, 1, 2])
b = pd.DataFrame({"y": [10, 20, 30]}, index=[1, 2, 3])
cote_a_cote = ...
communs = ...

verifier("C2", "4 lignes et 2 colonnes, avec 2 NaN", lambda: isinstance(cote_a_cote, pd.DataFrame) and cote_a_cote.shape == (4, 2) and cote_a_cote.isna().sum().sum() == 2)
verifier("C2", "inner : les étiquettes 1 et 2 seulement", lambda: isinstance(communs, pd.DataFrame) and list(communs.index) == [1, 2] and communs.isna().sum().sum() == 0)

**✏️ Exercice C3 — Joindre deux tables**

Avec `clients` et `achats` (cellule de cours) :
`r` : la jointure **inner** de `clients` (clé `id`) et `achats` (clé `client_id`).

<details>
<summary>💡 Indice 1 · la piste</summary>

`left_on` nomme la clé de gauche, `right_on` celle de droite ; `inner` est le comportement par défaut.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`pd.merge(clients, achats, left_on="id", right_on="client_id")`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

3 lignes : Ali (deux achats) et Driss (un achat).

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
r = ...

verifier("C3", "3 lignes et 6 colonnes", lambda: isinstance(r, pd.DataFrame) and r.shape == (3, 6))
verifier("C3", "Ali deux fois, puis Driss", lambda: list(r["nom"]) == ["Ali", "Ali", "Driss"])

**✏️ Exercice C4 — Quel how ?**

Le **nombre de lignes** de la jointure de `clients` et `achats` selon `how` :
`n_inner`, `n_left`, `n_right` et `n_outer`.

<details>
<summary>💡 Indice 1 · la piste</summary>

Un `merge` par valeur de `how`, puis `len`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`len(pd.merge(clients, achats, left_on="id", right_on="client_id", how="left"))`, et de même pour les autres.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

3, 4, 4 et 5.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
n_inner = ...
n_left = ...
n_right = ...
n_outer = ...

verifier("C4", "inner : 3 lignes", lambda: n_inner == 3)
verifier("C4", "left : 4 lignes, right : 4 lignes", lambda: n_left == 4 and n_right == 4)
verifier("C4", "outer : 5 lignes", lambda: n_outer == 5)

**✏️ Exercice C5 — Passagers et billets**

Avec `pas` et `bil` :
1. `full` : tous les **passagers**, avec leur billet quand il existe, et une **colonne d'audit** (`how="left"`, `indicator=True`)
2. `n_sans_billet` : le nombre de passagers **sans billet**

<details>
<summary>💡 Indice 1 · la piste</summary>

La clé est `PassengerId` ; `indicator=True` ajoute la colonne `_merge`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`full = pd.merge(pas, bil, on="PassengerId", how="left", indicator=True)` ; `n_sans_billet = (full["_merge"] == "left_only").sum()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

891 lignes ; 19 passagers sans billet.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
full = ...
n_sans_billet = ...

verifier("C5", "891 passagers gardés", lambda: isinstance(full, pd.DataFrame) and len(full) == 891 and "_merge" in full.columns)
verifier("C5", "19 passagers sans billet", lambda: n_sans_billet == 19)
verifier("C5", "Fare est NaN pour ces 19 passagers", lambda: full["Fare"].isna().sum() == 19)

**✏️ Exercice C6 — Les pièges du merge**

La table `dup` répète trois clés de `bil`. Calculez :
1. `n_dup` : le nombre de lignes de `pd.merge(pas, dup, on="PassengerId", how="left")` (piège silencieux)
2. `erreur_validate` : le **nom de l'exception** levée avec `validate="one_to_one"` (aide : la fonction `nom_erreur`)

<details>
<summary>💡 Indice 1 · la piste</summary>

`nom_erreur(f)` exécute `f()` et renvoie le nom de l'exception, ou `None`. Pour `f`, une fonction sans argument : `lambda: ...`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`n_dup = len(pd.merge(pas, dup, on="PassengerId", how="left"))` ; `erreur_validate = nom_erreur(lambda: pd.merge(pas, dup, on="PassengerId", validate="one_to_one"))`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

894 lignes au lieu de 891 ; l'exception s'appelle `MergeError`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
dup = pd.concat([bil, bil.head(3)])


def nom_erreur(f):
    """Nom de l'exception levée par f(), ou None si tout va bien."""
    try:
        f()
        return None
    except Exception as e:
        return type(e).__name__


n_dup = ...
erreur_validate = ...

verifier("C6", "894 lignes : le résultat a grossi", lambda: n_dup == 894)
verifier("C6", "validate lève une MergeError", lambda: erreur_validate == "MergeError")

---
## Partie D — Dater
_Diapos 28 à 34._

**Cours en bref**
- Une date écrite en texte n'est pas une date : `pd.to_datetime(s, format="%d/%m/%Y")` la convertit en type **`datetime64`**. **Donnez toujours le format** : `03/02/2026` est le 3 février (jour d'abord) ou le 2 mars (mois d'abord).
- L'accesseur **`.dt`** lit les morceaux : `.dt.year`, `.dt.month`, `.dt.day`, `.dt.day_of_week` (0 = lundi, 6 = dimanche), `.dt.day_name()`.
- Avec les dates en **index** (`set_index("Date")`), on découpe par texte : `loc["2026-02"]` (un mois), `["2026-01-03":"2026-01-06"]` (bornes incluses), `loc["2026"]` (une année). Attention : `ventes["2026"]` lève une `KeyError`.
- `resample("ME").sum()` regroupe **par période** (`ME` : fin de mois, `W` : semaine ; l'ancien alias `"M"` est refusé par Pandas 3).
- `shift(1)` donne la **veille**, `diff()` la **variation**, `rolling(7).mean()` la **moyenne glissante** sur 7 jours (les 6 premières valeurs sont `NaN`).

In [ ]:
def ventes_quotidiennes():
    """90 jours de ventes d'une librairie universitaire, fabriquées par une formule (sans hasard)."""
    dates = pd.date_range("2026-01-01", periods=90, freq="D")
    i = np.arange(90)
    effet = np.array([0, 2, 3, 4, 10, 18, 6])[dates.day_of_week]        # le week-end vend plus
    v = (60 + 0.4 * i + 3 * effet + np.round(4 * np.sin(i * 1.3))).astype(int)
    return pd.DataFrame({"Date": dates, "Ventes": v})


VENTES = ventes_quotidiennes()
print(VENTES.dtypes.to_dict())
print(VENTES.head(3))

**✏️ Exercice D1 — Du texte à une date**

Avec `s` (trois dates écrites jour/mois/année) :
1. `d` : `s` converti en dates avec le **format** `"%d/%m/%Y"`
2. `autre` : le texte `"03/02/2026"` lu **mois d'abord** (`"%m/%d/%Y"`), sous forme d'une Series d'un élément

<details>
<summary>💡 Indice 1 · la piste</summary>

`pd.to_datetime(serie, format=...)`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`d = pd.to_datetime(s, format="%d/%m/%Y")` ; `autre = pd.to_datetime(pd.Series(["03/02/2026"]), format="%m/%d/%Y")`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`d[0]` est le 3 février 2026 ; `autre[0]` est le 2 mars 2026.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
s = pd.Series(["03/02/2026", "15/02/2026", "28/02/2026"])
d = ...
autre = ...

verifier("D1", "d est de type date", lambda: isinstance(d, pd.Series) and pd.api.types.is_datetime64_any_dtype(d))
verifier("D1", "jour d'abord : le 3 février", lambda: d.iloc[0].month == 2 and d.iloc[0].day == 3)
verifier("D1", "mois d'abord : le 2 mars", lambda: autre.iloc[0].month == 3 and autre.iloc[0].day == 2)

**✏️ Exercice D2 — Les morceaux d'une date**

Avec `d` (exercice précédent) :
1. `mois` : le **mois** de chaque date
2. `jours_semaine` : le **jour de la semaine** (0 = lundi), avec `day_of_week`
3. `noms_jours` : le **nom du jour** (en anglais)

<details>
<summary>💡 Indice 1 · la piste</summary>

L'accesseur `.dt` donne accès aux morceaux.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`d.dt.month` ; `d.dt.day_of_week` ; `d.dt.day_name()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Le 3 février 2026 est un mardi (1), le 15 un dimanche (6), le 28 un samedi (5).

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
mois = ...
jours_semaine = ...
noms_jours = ...

verifier("D2", "février pour les trois dates", lambda: list(mois) == [2, 2, 2])
verifier("D2", "mardi, dimanche, samedi : 1, 6, 5", lambda: list(jours_semaine) == [1, 6, 5])
verifier("D2", "Tuesday, Sunday, Saturday", lambda: list(noms_jours) == ["Tuesday", "Sunday", "Saturday"])

**✏️ Exercice D3 — Découper par période**

1. `ventes` : `VENTES` avec la colonne `Date` en **index** (`set_index`)
2. `fevrier` : les lignes de **février 2026**
3. `plage` : les lignes **du 3 au 6 janvier 2026** (bornes incluses)

<details>
<summary>💡 Indice 1 · la piste</summary>

Une fois `Date` en index, on découpe avec des textes.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`ventes = VENTES.set_index("Date")` ; `ventes.loc["2026-02"]` ; `ventes["2026-01-03":"2026-01-06"]`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

28 lignes en février, 4 lignes pour la plage.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
ventes = ...
fevrier = ...
plage = ...

verifier("D3", "ventes est indexé par des dates", lambda: isinstance(ventes, pd.DataFrame) and isinstance(ventes.index, pd.DatetimeIndex))
verifier("D3", "28 lignes en février", lambda: isinstance(fevrier, pd.DataFrame) and len(fevrier) == 28)
verifier("D3", "4 lignes du 3 au 6 janvier", lambda: isinstance(plage, pd.DataFrame) and len(plage) == 4)

**✏️ Exercice D4 — Regrouper par période**

Avec `ventes` :
1. `mois_ventes` : le **total des ventes de chaque mois** (`resample("ME")`)
2. `semaines_ventes` : le **total de chaque semaine** (`resample("W")`)

<details>
<summary>💡 Indice 1 · la piste</summary>

`resample` s'applique à la colonne `Ventes` ; on choisit ensuite la statistique.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`ventes["Ventes"].resample("ME").sum()` ; `ventes["Ventes"].resample("W").sum()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Janvier : 2650 ; février : 2679 ; mars : 3311. Quatorze semaines.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
mois_ventes = ...
semaines_ventes = ...

verifier("D4", "2650, 2679 et 3311 ventes par mois", lambda: list(mois_ventes) == [2650, 2679, 3311])
verifier("D4", "quatorze totaux hebdomadaires", lambda: len(semaines_ventes) == 14)

**✏️ Exercice D5 — La veille et la moyenne glissante**

Sur `ventes5` (une copie fraîche de `VENTES`, indexée par la date), ajoutez deux colonnes :
1. `Veille` : la valeur de **la veille** (`shift`)
2. `Moyenne7` : la **moyenne glissante sur 7 jours** (`rolling`)

<details>
<summary>💡 Indice 1 · la piste</summary>

`shift(1)` décale d'une ligne ; `rolling(7).mean()`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`ventes["Veille"] = ventes["Ventes"].shift(1)` ; `ventes["Moyenne7"] = ventes["Ventes"].rolling(7).mean()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`Veille` vaut `NaN` au premier jour ; `Moyenne7` vaut `NaN` pendant 6 jours, puis environ 79,9.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
ventes5 = VENTES.set_index("Date")
ventes5["Veille"] = ...
ventes5["Moyenne7"] = ...

verifier("D5", "la veille du 2 janvier : 72", lambda: pd.api.types.is_float_dtype(ventes5["Veille"]) and ventes5["Veille"].iloc[1] == 72)
verifier("D5", "six NaN au début de la moyenne glissante", lambda: pd.api.types.is_float_dtype(ventes5["Moyenne7"]) and ventes5["Moyenne7"].iloc[:6].isna().all())
verifier("D5", "première moyenne sur 7 jours : 79,9", lambda: abs(ventes5["Moyenne7"].iloc[6] - 79.857) < 0.01)

**✏️ Exercice D6 — À vous : le meilleur jour**

1. `par_jour` : la **moyenne des ventes par jour de la semaine** (clé : `ventes.index.day_of_week`)
2. `meilleur_jour` : le **numéro du jour** où l'on vend le plus (`idxmax`)

<details>
<summary>💡 Indice 1 · la piste</summary>

Un `groupby` dont la clé est le jour de la semaine, lu dans l'index.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`ventes["Ventes"].groupby(ventes.index.day_of_week).mean()` ; `par_jour.idxmax()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Le samedi (5), avec 131 ventes en moyenne.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
par_jour = ...
meilleur_jour = ...

verifier("D6", "sept moyennes, du lundi au dimanche", lambda: isinstance(par_jour, pd.Series) and par_jour.shape == (7,))
verifier("D6", "le samedi vend 131 en moyenne", lambda: abs(par_jour[5] - 131.08) < 0.1)
verifier("D6", "meilleur jour : le samedi (5)", lambda: meilleur_jour == 5)

---
## Partie E — Appliquer
_Diapos 35 et 36._

**Cours en bref**
- `Series.map({...})` **traduit** chaque valeur avec un dictionnaire.
- `Series.apply(f)` appelle une **fonction Python** sur chaque valeur : très souple, mais **beaucoup plus lent** qu'une opération vectorisée (`s * 2`). On **vectorise d'abord**.
- `groupby(...).transform("median")` renvoie **une valeur par ligne** (la médiane de la classe de chaque passager) : on peut l'utiliser directement dans `fillna`. `agg` renvoie une ligne par groupe.

**✏️ Exercice E1 — map et apply**

1. `sexe_court` : la colonne `Sex` avec `male` → `"M"` et `female` → `"F"` (`map`)
2. `fare_double` : le tarif **doublé** avec `apply` et une fonction `lambda`
3. `meme_resultat` : `True` si `fare_double` est égal à `df["Fare"] * 2` (la version vectorisée, plus rapide)

<details>
<summary>💡 Indice 1 · la piste</summary>

`map` prend un dictionnaire ; `apply` prend une fonction.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df["Sex"].map({"male": "M", "female": "F"})` ; `df["Fare"].apply(lambda x: x * 2)` ; `np.allclose(fare_double, df["Fare"] * 2)`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

577 `"M"` et 314 `"F"` ; les deux méthodes donnent le même résultat.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
sexe_court = ...
fare_double = ...
meme_resultat = ...

verifier("E1", "577 M et 314 F", lambda: isinstance(sexe_court, pd.Series) and (sexe_court == "M").sum() == 577 and (sexe_court == "F").sum() == 314)
verifier("E1", "tarif doublé", lambda: isinstance(fare_double, pd.Series) and abs(fare_double.iloc[0] - 14.5) < 0.001)
verifier("E1", "apply et la version vectorisée donnent la même chose", lambda: meme_resultat is True or meme_resultat is np.True_)

**✏️ Exercice E2 — Remplir par la médiane de la classe**

1. `med_classe` : pour chaque passager, la **médiane d'âge de sa classe** (`groupby` puis `transform`)
2. `age_rempli` : `df["Age"]` avec les trous remplis par `med_classe` (**sans modifier `df`**)

<details>
<summary>💡 Indice 1 · la piste</summary>

`transform("median")` renvoie une valeur par ligne ; `fillna` accepte une Series.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`med_classe = df.groupby("Pclass")["Age"].transform("median")` ; `age_rempli = df["Age"].fillna(med_classe)`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Le passager 5, de 3e classe, reçoit 24 ans, la médiane de sa classe.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
med_classe = ...
age_rempli = ...

verifier("E2", "une valeur par passager", lambda: isinstance(med_classe, pd.Series) and len(med_classe) == 891 and med_classe.iloc[0] == 24)
verifier("E2", "plus aucun trou ; le passager 5 reçoit 24", lambda: isinstance(age_rempli, pd.Series) and age_rempli.isna().sum() == 0 and age_rempli.iloc[5] == 24)
verifier("E2", "df n'a pas été modifié : 177 trous", lambda: isinstance(age_rempli, pd.Series) and df["Age"].isna().sum() == 177)

---
## 📚 Pour aller plus loin
- GroupBy : https://pandas.pydata.org/docs/user_guide/groupby.html
- Réorganiser (pivot, melt) : https://pandas.pydata.org/docs/user_guide/reshaping.html
- Fusionner, joindre, concaténer : https://pandas.pydata.org/docs/user_guide/merging.html
- Séries temporelles : https://pandas.pydata.org/docs/user_guide/timeseries.html
- Notes de version de Pandas 3.0 : https://pandas.pydata.org/docs/whatsnew/v3.0.0.html

**Et maintenant :** le **mini-projet** (notebook `mini-projet-04`) reprend ces gestes sur **votre tableau propre** de la séance 3 et produit `s4_titanic_enrichi.csv` et `s4_synthese.csv`.